# Hypothesis 2 Validation
---

## Hypothesis
Increasing dataset size and diversity improves accuracy.

## Objectives
- Train the model on a smaller subset (10% of the dataset).
- Train the model on the full dataset.
- Compare performance metrics to validate Hypothesis 2.

## Inputs
- Animal species dataset (already split into train/val/test).
- Preprocessing and augmentation pipeline from earlier notebooks.

## Outputs
- Accuracy/Loss curves for both runs.
- Comparison table (subset vs full dataset).
- Conclusion on whether Hypothesis 2 is supported.
  
---

### Import libraries

In [7]:
import os, shutil, random
import pandas as pd
import matplotlib.pyplot as plt
import joblib as joblib
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam


## Set working directory

In [2]:
cwd = os.getcwd()
os.chdir('/workspaces/Animal_detection_camera')
print("You set a new current directory")

work_dir = os.getcwd()
work_dir

You set a new current directory


'/workspaces/Animal_detection_camera'

## Set input directories
Set train, validation and test paths

In [3]:
my_data_dir = 'inputs/datasets/animals/image'
train_path = my_data_dir + '/train'
val_path = my_data_dir + '/validation'
test_path = my_data_dir + '/test'

## Set output directory

In [4]:
version = 'v1'
file_path = f'outputs/{version}'
os.makedirs(file_path, exist_ok=True)

if 'outputs' in os.listdir(work_dir) and version in os.listdir(work_dir + '/outputs'):
    print('Old version is already available, create a new version.')
else:
    os.makedirs(name=file_path, exist_ok=True)

Old version is already available, create a new version.


## Set labels

In [5]:
labels = os.listdir(train_path)

print(
    f"Project Labels: {labels}"
)

Project Labels: ['lemur', 'snake', 'elephant', 'frog', 'chimpanzee', 'chinchilla', 'flamingo', 'mongoose', 'ostrich', 'ferret', 'camel', 'bee', 'mole', 'penguin', 'leopard', 'hawk', 'hedgehog', 'walrus', 'falcon', 'grasshopper', 'beaver', 'antelope', 'giraffe', 'duck', 'lizard', 'crab', 'goose', 'gorilla', 'jaguar', 'sheep', 'lynx', 'butterfly', 'panda', 'goat', 'deer', 'peacock', 'dog', 'whale', 'kangaroo', 'seal', 'cheetah', 'cow', 'iguana', 'hippopotamus', 'fox', 'cat', 'donkey', 'raccoon', 'blackbird', 'buffalo', 'koala', 'crocodile', 'dolphin', 'hyena', 'porcupine', 'bear', 'squid', 'spider', 'eagle', 'bison', 'owl', 'otter', 'snail', 'wolf']


## Set Image Shape

In [8]:
image_shape = joblib.load(filename=f"outputs/{version}/image_shape.pkl")
image_shape = (128, 128, 3)
print("Using image shape:", image_shape)

Using image shape: (128, 128, 3)


---

# Number of images in train, test and validation data
---

## Create subset


In [10]:
def create_subset(original_dir, subset_dir, fraction=0.1):
    os.makedirs(subset_dir, exist_ok=True)
    for class_name in os.listdir(original_dir):
        class_path = os.path.join(original_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        subset_class_path = os.path.join(subset_dir, class_name)
        os.makedirs(subset_class_path, exist_ok=True)

        files = os.listdir(class_path)
        sample_size = max(1, int(len(files) * fraction))
        sampled_files = random.sample(files, sample_size)

        for f in sampled_files:
            shutil.copy(os.path.join(class_path, f), os.path.join(subset_class_path, f))

create_subset("inputs/datasets/animals/image/train",
    "inputs/datasets/animals/small-data",
    fraction=0.1)


## Data Generators 
Small vs Full Datasets

In [11]:
datagen = ImageDataGenerator(rescale=1./255)

train_small_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/small-data", target_size=(128,128), batch_size=32, class_mode='categorical')

train_full_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/train", target_size=(128,128), batch_size=32, class_mode='categorical')

val_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/validation", target_size=(128,128), batch_size=32, class_mode='categorical')


Found 997 images belonging to 64 classes.
Found 10050 images belonging to 64 classes.
Found 1400 images belonging to 64 classes.
